In [1]:
from pathlib import Path
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from subprocess import run, PIPE
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

In [5]:
DATA_PATH = Path("/net/people/plgrid/plgjedrzejkusnierz/big_storage/data/WildSVDD")

In [6]:
files = [p for p in DATA_PATH.rglob("*") if p.is_file() and p.suffix.lower() == ".flac"]

def ffprobe_duration(path: Path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    proc = run(cmd, stdout=PIPE, stderr=PIPE, text=True)
    if proc.returncode != 0 or not proc.stdout.strip():
        return {"error": True, "filepath": str(path), "error_msg": proc.stderr.strip() or "no-duration"}
    try:
        dur = float(proc.stdout.strip())
        return {"filename": path.stem, "filepath": str(path), "duration": int(dur)}
    except Exception as e:
        return {"error": True, "filepath": str(path), "error_msg": repr(e)}

results = []
failed = []
max_workers = min(32, max(4, (os.cpu_count() or 4) * 2))
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(ffprobe_duration, p): p for p in files}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if res.get("error"):
            failed.append((res["filepath"], res["error_msg"]))
        else:
            results.append(res)

df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed, columns=["filepath", "error"])
print("success:", len(df), "failed:", len(failed_df))

  0%|          | 0/515 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'ffprobe'

In [4]:
df.describe()

ValueError: Cannot describe a DataFrame without columns

In [ ]:
sns.histplot(df, x='duration', bins=100)

In [ ]:
filtered_df = df[df['duration'] < 1000]

In [ ]:
sns.histplot(filtered_df, x='duration', bins=100)

In [ ]:
df[df['duration'] > 500].count()

In [ ]:
df[df['duration'] > 500]['filepath']

In [ ]:
files_to_remove = df[df['duration'] > 500]['filepath']
for filepath in files_to_remove:
    if os.path.exists(filepath):
        os.remove(filepath)
        print(f"Removed: {filepath}")
    else:
        print(f"File not found: {filepath}")

In [ ]:
csv_path = Path("/mnt/data/kusnierz/audio-data/WildSVDD/train.csv")
downloads_path = Path("/mnt/data/kusnierz/audio-data/WildSVDD/downloads")

In [ ]:
from pathlib import Path
import pandas as pd
import json
from collections import Counter

# Paths
wildsvdd_dir = Path("/mnt/data/kusnierz/audio-data/WildSVDD")
downloads_dir = wildsvdd_dir / "downloads"
train_csv = wildsvdd_dir / "train.csv"
test_a_csv = wildsvdd_dir / "test_A.csv"
test_b_csv = wildsvdd_dir / "test_B.csv"
singfake_csv = Path("/mnt/data/kusnierz/audio-data/SingFake/singfake.csv")

# Read CSVs
df_train = pd.read_csv(train_csv)
df_test_a = pd.read_csv(test_a_csv)
df_test_b = pd.read_csv(test_b_csv)
df_singfake = pd.read_csv(singfake_csv, header=None, names=["split", "label", "lang", "artist", "title", "unknown", "url"])

# Helper: get unique URLs from SingFake
singfake_urls = set(df_singfake['url'])

def filter_singfake_overlap(df):
    return df[~df['Url'].isin(singfake_urls)]

def get_expected_stats(df, split_name):
    filtered = filter_singfake_overlap(df)
    return filtered['Bonafide Or Deepfake'].value_counts(), filtered.shape[0]

# Get expected stats for each split
splits = {
    'Training': df_train,
    'TestA': df_test_a,
    'TestB': df_test_b
}
expected_stats = {}
for split, df in splits.items():
    counts, total = get_expected_stats(df, split)
    expected_stats[split] = {'counts': counts.to_dict(), 'total': total}

# List downloaded files and parse their type and split
files = [f for f in downloads_dir.glob("*.flac") if f.is_file()]
with open(downloads_dir / "metadata.json", "r") as f:
    metadata = json.load(f)
filename_to_url = {entry["filename"]: entry["url"] for entry in metadata if "filename" in entry and "url" in entry}
filename_to_set = {entry["filename"]: entry.get("set", "Unknown") for entry in metadata if "filename" in entry}

filtered_files = [f for f in files if filename_to_url.get(f.name) not in singfake_urls]

def parse_type_from_name(name):
    if "deepfake" in name.lower():
        return "deepfake"
    elif "bonafide" in name.lower():
        return "bonafide"
    else:
        return "unknown"

def parse_split_from_name(name):
    # Use metadata if available
    return filename_to_set.get(name, "Unknown")

# Gather downloaded stats per split
downloaded_stats = {}
for split in ['Training', 'TestA', 'TestB']:
    split_files = [f for f in filtered_files if parse_split_from_name(f.name).lower() == split.lower()]
    types = [parse_type_from_name(f.name) for f in split_files]
    downloaded_stats[split] = dict(Counter(types))
    downloaded_stats[split]['total'] = len(split_files)

# Print comparison
for split in ['Training', 'TestA', 'TestB']:
    print(f"--- {split} ---")
    print("Expected (CSV, filtered):", expected_stats[split]['counts'], f"Total: {expected_stats[split]['total']}")
    print("Downloaded (filtered):", {k: v for k, v in downloaded_stats[split].items() if k != 'total'}, f"Total: {downloaded_stats[split]['total']}")
    print()


In [ ]:
# Language distribution analysis (filtered for SingFake overlaps)

import matplotlib.pyplot as plt
import seaborn as sns

# Combine all filtered data for general distribution
all_filtered = pd.concat([
    filter_singfake_overlap(df_train).assign(split='Training'),
    filter_singfake_overlap(df_test_a).assign(split='TestA'),
    filter_singfake_overlap(df_test_b).assign(split='TestB')
], ignore_index=True)

# General language distribution
lang_counts = all_filtered['Language'].value_counts()
plt.figure(figsize=(10,5))
ax = sns.barplot(x=lang_counts.index, y=lang_counts.values)
plt.xticks(rotation=45)
plt.title('Language Distribution (General, WildSVDD filtered)')
plt.ylabel('Count')
plt.xlabel('Language')
plt.tight_layout()
for i, v in enumerate(lang_counts.values):
    ax.text(i, v + max(lang_counts.values)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
plt.show()

# Language distribution by split
split_lang_counts = all_filtered.groupby('split')['Language'].value_counts().unstack(fill_value=0)
split_lang_counts.plot(kind='bar', stacked=True, figsize=(12,6))
plt.title('Language Distribution by Split (WildSVDD filtered)')
plt.ylabel('Count')
plt.xlabel('Split')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()